# 03 — Exploratory Data Analysis (EDA)

**Goal:** Explore churn patterns, feature distributions, and relationships through visualisations and statistical summaries.

---

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data_preprocessing import load_raw_data, clean_data, TARGET_COL, ID_COL

# Style
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
FIGURES_DIR = Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Load and clean
df_raw = load_raw_data()
df = clean_data(df_raw)
print(f"Working with {len(df)} rows, {len(df.columns)} columns")
print(f"Churn rate: {df[TARGET_COL].mean():.4f} ({df[TARGET_COL].mean()*100:.1f}%)")

Loaded WA_Fn-UseC_-Telco-Customer-Churn.csv: 7043 rows, 21 columns
Working with 7043 rows, 21 columns
Churn rate: 0.2654 (26.5%)


## 3.1 Churn Class Distribution

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
churn_counts = df[TARGET_COL].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['No Churn (0)', 'Churn (1)'], churn_counts.values, color=colors, edgecolor='white', linewidth=1.5)
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, f'{v} ({v/len(df)*100:.1f}%)', ha='center', fontweight='bold', fontsize=12)
axes[0].set_title('Churn Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(churn_counts.values, labels=['No Churn', 'Churn'], colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Churn Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'churn_distribution.png', dpi=150)
plt.show()
print("\nObservation: The dataset shows moderate class imbalance (~26.5% churn). "
      "We will need to account for this during model training.")


Observation: The dataset shows moderate class imbalance (~26.5% churn). We will need to account for this during model training.


C:\Users\Sandhya More\AppData\Local\Temp\ipykernel_20644\1923598999.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.2 Churn Rate by Contract Type

In [4]:
if 'Contract' in df.columns:
    contract_churn = df.groupby('Contract')[TARGET_COL].agg(['mean', 'count']).reset_index()
    contract_churn.columns = ['Contract', 'Churn Rate', 'Count']
    contract_churn = contract_churn.sort_values('Churn Rate', ascending=False)
    
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(contract_churn['Contract'], contract_churn['Churn Rate'],
                  color=['#e74c3c', '#f39c12', '#2ecc71'], edgecolor='white', linewidth=1.5)
    for bar, rate, count in zip(bars, contract_churn['Churn Rate'], contract_churn['Count']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{rate:.1%}\n(n={count})', ha='center', fontsize=11, fontweight='bold')
    ax.set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')
    ax.set_ylabel('Churn Rate')
    ax.set_ylim(0, 0.65)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'churn_by_contract.png', dpi=150)
    plt.show()
    print("Observation: Month-to-month customers churn at significantly higher rates "
          "than those on longer contracts.")

C:\Users\Sandhya More\AppData\Local\Temp\ipykernel_20644\2121148888.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Observation: Month-to-month customers churn at significantly higher rates than those on longer contracts.


## 3.3 Tenure Distribution by Churn Status

In [5]:
if 'tenure' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        subset = df[df[TARGET_COL] == label]['tenure']
        name = 'Churn' if label == 1 else 'No Churn'
        axes[0].hist(subset, bins=30, alpha=0.6, color=color, label=name, edgecolor='white')
    axes[0].set_title('Tenure Distribution by Churn Status', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Tenure (months)')
    axes[0].set_ylabel('Count')
    axes[0].legend()
    
    # Box plot
    df['Churn_Label'] = df[TARGET_COL].map({0: 'No Churn', 1: 'Churn'})
    sns.boxplot(data=df, x='Churn_Label', y='tenure', palette=['#2ecc71', '#e74c3c'], ax=axes[1])
    axes[1].set_title('Tenure by Churn Status', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('')
    axes[1].set_ylabel('Tenure (months)')
    
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'tenure_by_churn.png', dpi=150)
    plt.show()
    df.drop(columns=['Churn_Label'], inplace=True)
    
    print(f"\nMedian tenure — No Churn: {df[df[TARGET_COL]==0]['tenure'].median():.0f} months | "
          f"Churn: {df[df[TARGET_COL]==1]['tenure'].median():.0f} months")
    print("Observation: Churned customers tend to have much shorter tenures. "
          "New customers are at higher risk.")

C:\Users\Sandhya More\AppData\Local\Temp\ipykernel_20644\2912927378.py:16: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x='Churn_Label', y='tenure', palette=['#2ecc71', '#e74c3c'], ax=axes[1])



Median tenure — No Churn: 38 months | Churn: 10 months
Observation: Churned customers tend to have much shorter tenures. New customers are at higher risk.


C:\Users\Sandhya More\AppData\Local\Temp\ipykernel_20644\2912927378.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.4 Monthly Charges by Churn Status

In [6]:
if 'MonthlyCharges' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # KDE plot
    for label, color, name in [(0, '#2ecc71', 'No Churn'), (1, '#e74c3c', 'Churn')]:
        subset = df[df[TARGET_COL] == label]['MonthlyCharges']
        subset.plot.kde(ax=axes[0], color=color, label=name, linewidth=2)
    axes[0].set_title('Monthly Charges Distribution', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Monthly Charges ($)')
    axes[0].legend()
    axes[0].set_xlim(0, 130)
    
    # Violin plot
    df['Churn_Label'] = df[TARGET_COL].map({0: 'No Churn', 1: 'Churn'})
    sns.violinplot(data=df, x='Churn_Label', y='MonthlyCharges',
                   palette=['#2ecc71', '#e74c3c'], ax=axes[1], inner='box')
    axes[1].set_title('Monthly Charges by Churn Status', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('')
    axes[1].set_ylabel('Monthly Charges ($)')
    
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'monthly_charges_by_churn.png', dpi=150)
    plt.show()
    df.drop(columns=['Churn_Label'], inplace=True)
    
    print(f"\nMean Monthly Charges — No Churn: ${df[df[TARGET_COL]==0]['MonthlyCharges'].mean():.2f} | "
          f"Churn: ${df[df[TARGET_COL]==1]['MonthlyCharges'].mean():.2f}")
    print("Observation: Churned customers tend to have higher monthly charges, "
          "possibly correlated with fibre optic internet service.")

C:\Users\Sandhya More\AppData\Local\Temp\ipykernel_20644\2108744419.py:15: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x='Churn_Label', y='MonthlyCharges',



Mean Monthly Charges — No Churn: $61.27 | Churn: $74.44
Observation: Churned customers tend to have higher monthly charges, possibly correlated with fibre optic internet service.


C:\Users\Sandhya More\AppData\Local\Temp\ipykernel_20644\2108744419.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.5 Churn Rate by Key Categorical Features

In [7]:
cat_features = ['InternetService', 'PaymentMethod', 'PaperlessBilling',
                'PhoneService', 'Partner', 'Dependents', 'SeniorCitizen']
cat_features = [c for c in cat_features if c in df.columns]

n_cols = 2
n_rows = (len(cat_features) + 1) // 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    churn_rate = df.groupby(col)[TARGET_COL].mean().sort_values(ascending=False)
    churn_rate.plot(kind='bar', ax=axes[i], color='#3498db', edgecolor='white', linewidth=1.5)
    axes[i].set_title(f'Churn Rate by {col}', fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Churn Rate')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')
    axes[i].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    # Add value labels
    for bar in axes[i].patches:
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{bar.get_height():.1%}', ha='center', fontsize=9)

# Hide extra axes
for j in range(len(cat_features), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'churn_by_categorical.png', dpi=150)
plt.show()

C:\Users\Sandhya More\AppData\Local\Temp\ipykernel_20644\2271586221.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.6 Correlation Heatmap (Numeric Features)

In [8]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) > 1:
    fig, ax = plt.subplots(figsize=(10, 8))
    corr = df[numeric_cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, ax=ax, square=True, linewidths=0.5)
    ax.set_title('Correlation Matrix — Numeric Features', fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'correlation_heatmap.png', dpi=150)
    plt.show()
    print("Observation: TotalCharges and tenure are strongly correlated (expected). "
          "MonthlyCharges shows a positive correlation with churn.")

Observation: TotalCharges and tenure are strongly correlated (expected). MonthlyCharges shows a positive correlation with churn.


C:\Users\Sandhya More\AppData\Local\Temp\ipykernel_20644\4250038657.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.7 Service Usage and Churn

In [9]:
services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies']
services = [s for s in services if s in df.columns]

if services:
    fig, ax = plt.subplots(figsize=(10, 6))
    service_churn = {}
    for s in services:
        yes_churn = df[df[s] == 'Yes'][TARGET_COL].mean()
        no_churn = df[df[s] == 'No'][TARGET_COL].mean()
        service_churn[s] = {'With Service': yes_churn, 'Without Service': no_churn}
    
    sdf = pd.DataFrame(service_churn).T
    sdf.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
    ax.set_title('Churn Rate: With vs Without Service', fontsize=14, fontweight='bold')
    ax.set_ylabel('Churn Rate')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.legend(title='Service Status')
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'service_churn_comparison.png', dpi=150)
    plt.show()
    print("Observation: Customers WITHOUT protective services (security, backup, "
          "device protection, tech support) tend to have higher churn rates.")

Observation: Customers WITHOUT protective services (security, backup, device protection, tech support) tend to have higher churn rates.


C:\Users\Sandhya More\AppData\Local\Temp\ipykernel_20644\3628882101.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.8 EDA Summary

Key findings from exploratory analysis:

1. **Class imbalance:** ~26.5% churn rate — moderate imbalance that should be addressed in training.
2. **Contract type is highly predictive:** Month-to-month customers churn at ~42% vs ~3-11% for yearly contracts.
3. **Short tenure = higher risk:** Churned customers have median tenure of ~10 months vs ~38 months for retained.
4. **Higher charges correlate with churn:** Mean monthly charges are higher for churned customers.
5. **Service adoption matters:** Customers without protective services (security, backup, tech support) show higher churn.
6. **Payment method:** Electronic check users have the highest churn rate.
7. **Demographics:** Senior citizens show slightly higher churn; single customers (no partner/dependents) churn more.

**Reminder:** These are correlations observed in the data, not proven causal relationships.